# 2.1.1 — Discrete-time signals as a sum of shifted impulses

$$x[n] \;=\; \sum_{k=-\infty}^{+\infty} x[k]\,\delta[n-k]$$

You can read this one equation in several ways, and the convolution sum in 2.1.2 depends on each of them.
Each view below shows one reading:

| view | what's fixed | what varies | reading |
|---|---|---|---|
| **A** decomposition (Fig. 2.1) | — | — | a signal *is* a stack of scaled, shifted impulses |
| **B** build-up over **k** | step through k | n | add one term $x[k]\delta[n-k]$ at a time, so the signal assembles itself |
| **C** sifting at fixed **n** | n | k | for one output index, $\delta[n-k]$ as a function of **k** selects exactly $x[n]$ |
| **D** the $(k,n)$ matrix | — | both | rows = reading B, columns = reading C, column sums = the output |

Views B–D have a **kernel** dropdown. With $\delta[n]$ you get 2.1.1. With an impulse response
$h[n]$, the same picture becomes the convolution sum $y[n] = \sum_k x[k]\,h[n-k]$ (2.1.2):
- B becomes the *superposition of shifted responses* (Fig. 2.2–2.3).
- C becomes *flip and slide*.
- D becomes the convolution matrix.

Color legend used everywhere: <span style="color:#2a78d6">■ input x</span> ·
<span style="color:#eb6834">■ kernel δ / h</span> · <span style="color:#1baf7a">■ one term / product</span> ·
■ sum / output · <span style="color:#b5b4ac">■ context (ghost)</span>

In [3]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as w
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
from IPython.display import display

plt.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': '#52514e',
    'xtick.color': '#898781', 'ytick.color': '#898781',
    'axes.titlesize': 10, 'axes.titlecolor': '#0b0b0b', 'axes.titlelocation': 'left',
    'grid.color': '#e1e0d9', 'grid.linewidth': 0.6,
    'font.size': 9,
})

C_X, C_H, C_TERM, C_SUM = '#2a78d6', '#eb6834', '#1baf7a', '#0b0b0b'
C_GHOST, C_MUTED, C_INK2 = '#d6d5ce', '#898781', '#52514e'
# negative <-> positive; deliberately not blue/orange, those mean x and h here
DIVERGING = LinearSegmentedColormap.from_list('div', ['#6250d6', '#f0efec', '#eda100'])

K = np.arange(-4, 5)    # every test signal lives on K (finite length)
N = np.arange(-8, 13)   # time axis shown (wide enough for x * h to finish)


def delta(n):
  return (np.asarray(n) == 0).astype(float)


def u(n):
  return (np.asarray(n) >= 0).astype(float)


def from_values(vals):
  # signal defined by its values on K, zero elsewhere
  vals = np.asarray(vals, float)

  def x(n):
    n = np.atleast_1d(n)
    out = np.zeros(n.shape)
    m = (n >= K[0]) & (n <= K[-1])
    out[m] = vals[n[m] - K[0]]
    return out

  return x


SIGNALS = {
    'Fig. 2.1 style': from_values([0, 0, 1.0, -1.5, 2.0, 1.0, -0.5, 1.5, 0]),
    'rectangle': from_values([0, 0, 1, 1, 1, 1, 1, 0, 0]),
    'ramp': from_values(0.5 * np.maximum(K, 0)),
    'decaying exp': from_values(0.7 ** np.maximum(K, 0) * u(K)),
    'cosine': from_values(np.cos(np.pi * K / 4)),
    'random': from_values(np.round(np.random.default_rng(1).uniform(-2, 2, K.size), 1)),
}

# (symbol used in formulas, function)
KERNELS = {
    'δ[n]  — identity (§2.1.1)': ('\\delta', delta),
    'u[n]−u[n−3]  — 3-pt running sum': ('h', lambda n: u(n) - u(n - 3)),
    '0.6ⁿ u[n]  — leaky integrator': ('h', lambda n: 0.6 ** np.maximum(n, 0) * u(n)),
    'δ[n]−δ[n−1]  — first difference': ('h', lambda n: delta(n) - delta(n - 1)),
    'δ[n−2]  — delay by 2': ('h', lambda n: delta(n - 2)),
}


def shift(k, var='n'):
  # 'n-k' rendered for a concrete k:  k=-2 -> 'n+2', k=0 -> 'n'
  return var if k == 0 else f'{var}{-k:+d}'


def flipped(n0):
  # 'n0-k' rendered for a concrete n0
  return '-k' if n0 == 0 else f'{n0}-k'


def stem(ax, n, y, color, ms=5, lw=1.6, z=3):
  # nonzero samples as stems in `color`, zero samples as small muted dots
  n, y = np.asarray(n), np.asarray(y, float)
  nz = y != 0
  ax.vlines(n[nz], 0, y[nz], color=color, lw=lw, zorder=z)
  ax.plot(n[nz], y[nz], 'o', color=color, ms=ms, mec='#fcfcfb', mew=0.8, zorder=z + 1)
  ax.plot(n[~nz], y[~nz], 'o', color=C_GHOST, ms=2.5, zorder=z)


def frame(ax, n, ylim, xlabel=None):
  ax.axhline(0, color='#c3c2b7', lw=0.8, zorder=1)
  ax.set_xlim(n[0] - 0.7, n[-1] + 0.7)
  ax.set_xticks(n)
  ax.set_ylim(*ylim)
  ax.grid(axis='x')
  if xlabel:
    ax.set_xlabel(xlabel)


def ylim_for(*arrays, pad=1.3):
  m = max(np.max(np.abs(a)) for a in arrays) or 1
  return -m * pad, m * pad


def interactive_fig(**kw):
  # figure that is displayed only when we put it under the controls
  with plt.ioff():
    fig = plt.figure(**kw)
  fig.canvas.header_visible = False
  fig.canvas.toolbar_visible = False
  return fig


def play_slider(label, lo, hi, value, interval=700):
  slider = w.IntSlider(value=value, min=lo, max=hi, description=label,
                       continuous_update=True, layout=w.Layout(width='420px'))
  play = w.Play(value=value, min=lo, max=hi, interval=interval)
  w.jslink((play, 'value'), (slider, 'value'))
  return play, slider


def dropdown(options, description):
  return w.Dropdown(options=list(options), description=description,
                    layout=w.Layout(width='340px'))

## A — the decomposition itself (Oppenheim Fig. 2.1)

Each row is one term $x[k]\,\delta[n-k]$: a single impulse at $n=k$ scaled by the sample value $x[k]$.
The gray ghost behind each row is the full signal, so you can see where that term comes from.
Adding the rows column by column gives back $x[n]$ in the last row.

In [4]:
figA = interactive_fig(figsize=(8.5, 11))
axesA = figA.subplots(len(K) + 2, 1, sharex=True)
figA.subplots_adjust(left=0.07, right=0.80, top=0.96, bottom=0.05, hspace=0.25)


def draw_A(name):
  x = SIGNALS[name]
  xn = x(N)
  lim = ylim_for(xn)
  terms = [x(k) * delta(N - k) for k in K]

  for ax in axesA:
    ax.clear()
    frame(ax, N, lim)
    ax.set_yticks([])
    ax.spines['left'].set_visible(False)

  stem(axesA[0], N, xn, C_X)
  axesA[0].set_title(f'x[n]  ({name})')

  for i, (ax, k, term) in enumerate(zip(axesA[1:-1], K, terms)):
    stem(ax, N, xn, C_GHOST, ms=3.5, lw=1, z=2)
    stem(ax, N, term, C_TERM)
    ax.text(1.02, 0.5, f'$x[{k}]\\,\\delta[{shift(k)}]$', transform=ax.transAxes,
            va='center', color=C_SUM, fontsize=10)
    ax.text(1.02, 0.12, f'= {x(k)[0]:g}·δ[{shift(k)}]', transform=ax.transAxes,
            va='center', color=C_MUTED, fontsize=8)
    ax.text(-0.03, 0.5, '=' if i == 0 else '+', transform=ax.transAxes,
            ha='center', va='center', fontsize=13, color=C_INK2)

  total = np.sum(terms, axis=0)
  assert np.allclose(total, xn)
  stem(axesA[-1], N, total, C_SUM)
  axesA[-1].text(1.02, 0.5, '$\\sum_k x[k]\\,\\delta[n-k] = x[n]$',
                 transform=axesA[-1].transAxes, va='center', fontsize=10)
  axesA[-1].text(-0.03, 0.5, '=', transform=axesA[-1].transAxes,
                 ha='center', va='center', fontsize=13, color=C_INK2)
  axesA[-1].set_xlabel('n')
  figA.canvas.draw_idle()


sigA = dropdown(SIGNALS, 'signal')
sigA.observe(lambda ch: draw_A(ch.new), names='value')
draw_A(sigA.value)
display(w.VBox([sigA, figA.canvas]))

## B — build it up one term at a time (iterate over **k**)

Step through $k$. At each step:
1. take the sample $x[k]$ (top),
2. turn it into its own sequence $x[k]\,\delta[n-k]$, an impulse placed at $n=k$ (middle),
3. add that sequence to the running sum (bottom).

Switch the kernel to an $h[n]$ and step 2 changes: each sample now produces a **scaled, shifted copy
of the impulse response**, $x[k]\,h[n-k]$. That is linearity and time invariance applied to the decomposition from A.
This is the LTI argument of 2.1.2, and the running sum becomes $y[n]$.

In [5]:
figB = interactive_fig(figsize=(8.5, 7))
axB = figB.subplots(3, 1)
figB.subplots_adjust(left=0.08, right=0.97, top=0.93, bottom=0.08, hspace=0.55)


def draw_B(*_):
  x = SIGNALS[sigB.value]
  sym, h = KERNELS[kerB.value]
  out = 'x' if sym == '\\delta' else 'y'
  k0 = kB.value
  xn = x(N)
  terms = {k: x(k) * h(N - k) for k in K}
  done = sum((terms[k] for k in K if k < k0), np.zeros(N.size))
  full = sum(terms.values())
  lim = ylim_for(xn, full, *terms.values())

  for ax in axB:
    ax.clear()

  # 1. pick x[k0]
  ax = axB[0]
  frame(ax, N, ylim_for(xn))
  stem(ax, N, xn, '#9ec5f4', ms=4, lw=1.2)
  stem(ax, [k0], x(k0), C_X, ms=7, lw=2.2)
  ax.set_title(f'1.  pick the sample  x[{k0}] = {x(k0)[0]:g}')

  # 2. the term it contributes
  ax = axB[1]
  frame(ax, N, lim)
  stem(ax, N, terms[k0], C_TERM)
  ax.set_title(f'2.  its contribution  $x[{k0}]\\,{sym}[{shift(k0)}]$'
               + ('' if sym == '\\delta' else f'   (copy of h, scaled by {x(k0)[0]:g}, moved to n={k0})'))

  # 3. running sum: previous terms (ink) + this term stacked on top (aqua) + final result (ghost)
  ax = axB[2]
  frame(ax, N, lim, 'n')
  stem(ax, N, full, C_GHOST, ms=3.5, lw=1, z=2)
  ax.vlines(N, done, done + terms[k0], color=C_TERM, lw=3, alpha=0.6, zorder=3)
  stem(ax, N, done + terms[k0], C_SUM, z=4)
  ax.set_title(f'3.  running sum  $\\sum_{{k={K[0]}}}^{{{k0}}} x[k]\\,{sym}[n-k]$'
               f'   → {out}[n] when k reaches {K[-1]}')
  figB.canvas.draw_idle()


sigB = dropdown(SIGNALS, 'signal')
kerB = dropdown(KERNELS, 'kernel')
playB, kB = play_slider('k', K[0], K[-1], K[0], interval=900)
for wd in (sigB, kerB, kB):
  wd.observe(draw_B, names='value')
draw_B()
display(w.VBox([w.HBox([sigB, kerB]), w.HBox([playB, kB]), figB.canvas]))

## C — sifting: fix **n**, look along **k**

Now hold the output index $n$ fixed and read the sum as a function of $k$:

$$x[n] = \sum_k \underbrace{x[k]}_{\text{row 1}}\;\underbrace{\delta[n-k]}_{\text{row 2, a function of }k}$$

As a function of $k$, $\delta[n-k]$ is a single impulse at $k=n$. The product (row 3) is zero everywhere except
there, so the sum over $k$ returns exactly $x[n]$. This is the **sifting property**. Slide $n$ and the output
(row 4) gets filled in one sample at a time.

Row 2 is really $\delta[k]$ **flipped** ($-k$) and then **shifted** by $n$. You can't see the flip with δ because δ is
symmetric. Switch the kernel to an asymmetric $h$ and the flip becomes visible. This is the *flip and slide* method of computing
$y[n]=\sum_k x[k]\,h[n-k]$.

In [6]:
figC = interactive_fig(figsize=(8.5, 8.5))
axC = figC.subplots(4, 1)
figC.subplots_adjust(left=0.08, right=0.97, top=0.94, bottom=0.07, hspace=0.6)


def draw_C(*_):
  x = SIGNALS[sigC.value]
  sym, h = KERNELS[kerC.value]
  out = 'x' if sym == '\\delta' else 'y'
  n0 = nC.value
  xk = x(N)
  hk = h(n0 - N)                       # as a function of k (N doubles as the k axis)
  prod = xk * hk
  y = np.array([np.sum(xk * h(m - N)) for m in N])

  for ax in axC:
    ax.clear()

  frame(axC[0], N, ylim_for(xk), None)
  stem(axC[0], N, xk, C_X)
  axC[0].set_title('x[k]   (input, as a function of k)')

  frame(axC[1], N, ylim_for(h(N)))
  stem(axC[1], N, hk, C_H)
  axC[1].axvline(n0, color=C_H, lw=0.8, ls=':', zorder=1)
  axC[1].set_title(f'${sym}[{flipped(n0)}]$   ({sym}[k] flipped, then slid to n = {n0})')

  frame(axC[2], N, ylim_for(prod, xk))
  stem(axC[2], N, prod, C_TERM)
  axC[2].set_title(f'$x[k]\\,{sym}[{flipped(n0)}]$   →   sum over k = {prod.sum():.3g}')
  axC[2].set_xlabel('k')

  ax = axC[3]
  frame(ax, N, ylim_for(y), 'n')
  stem(ax, N, y, C_GHOST, ms=3.5, lw=1, z=2)
  stem(ax, N[N <= n0], y[N <= n0], C_SUM)
  stem(ax, [n0], [y[N == n0][0]], C_TERM, ms=8, lw=2.4, z=5)
  ax.set_title(f'{out}[n]   filled in up to n = {n0};   {out}[{n0}] = {prod.sum():.3g}')
  figC.canvas.draw_idle()


sigC = dropdown(SIGNALS, 'signal')
kerC = dropdown(KERNELS, 'kernel')
playC, nC = play_slider('n', N[0], N[-1], -5, interval=700)
for wd in (sigC, kerC, nC):
  wd.observe(draw_C, names='value')
draw_C()
display(w.VBox([w.HBox([sigC, kerC]), w.HBox([playC, nC]), figC.canvas]))

## D — both readings at once: the $(k, n)$ matrix

Put every term in one 2-D array $M[k,n] = x[k]\,\delta[n-k]$ (or $x[k]\,h[n-k]$):

* a **row** $k$ is one term as a function of $n$, the thing B adds up (top panel),
* a **column** $n$ is the product as a function of $k$, the thing C sums (right panel),
* **summing each column** gives the output (bottom panel).

With δ the matrix is **diagonal**, which is why each output sample receives exactly one input sample.
With $h$ the diagonal turns into a band (each diagonal is constant along $n-k$, as in a Toeplitz matrix), and $y = Mx$ is
convolution written as a matrix-vector product. Hover over the heatmap to read values in the status bar.

In [7]:
figD = interactive_fig(figsize=(9.5, 8))
gs = figD.add_gridspec(3, 2, width_ratios=[4, 1.3], height_ratios=[1.1, 2.2, 1.1],
                       hspace=0.45, wspace=0.08)
axD_heat = figD.add_subplot(gs[1, 0])
axD_row = figD.add_subplot(gs[0, 0], sharex=axD_heat)
axD_out = figD.add_subplot(gs[2, 0], sharex=axD_heat)
axD_col = figD.add_subplot(gs[1, 1], sharey=axD_heat)
axD_txt = figD.add_subplot(gs[0, 1])
figD.subplots_adjust(left=0.07, right=0.97, top=0.95, bottom=0.07)
figD.canvas.toolbar_visible = True


def draw_D(*_):
  x = SIGNALS[sigD.value]
  sym, h = KERNELS[kerD.value]
  out = 'x' if sym == '\\delta' else 'y'
  k0, n0 = kD.value, nD.value
  M = x(K)[:, None] * h(N[None, :] - K[:, None])     # rows k, columns n
  y = M.sum(axis=0)
  vmax = np.abs(M).max() or 1
  ci = int(np.where(N == n0)[0][0])
  ri = int(np.where(K == k0)[0][0])

  for ax in (axD_heat, axD_row, axD_out, axD_col, axD_txt):
    ax.clear()

  # heatmap, k increasing downward like the stacked rows of Fig. 2.1
  ax = axD_heat
  ax.imshow(M, cmap=DIVERGING, vmin=-vmax, vmax=vmax, aspect='auto',
            extent=(N[0] - .5, N[-1] + .5, K[-1] + .5, K[0] - .5), interpolation='nearest')
  for i, k in enumerate(K):
    for j, n in enumerate(N):
      if M[i, j] != 0:
        ax.text(n, k, f'{M[i, j]:.2g}', ha='center', va='center', fontsize=6.5, color=C_SUM)
  ax.add_patch(Rectangle((N[0] - .5, k0 - .5), N.size, 1, fill=False, ec=C_TERM, lw=2))
  ax.add_patch(Rectangle((n0 - .5, K[0] - .5), 1, K.size, fill=False, ec=C_SUM, lw=2))
  ax.set_yticks(K)
  ax.set_ylabel('k  (input index)')
  ax.set_title(f'$M[k,n] = x[k]\\,{sym}[n-k]$', loc='left')
  plt.setp(ax.get_xticklabels(), visible=False)

  # one row: term k0 over n
  ax = axD_row
  frame(ax, N, ylim_for(M))
  stem(ax, N, M[ri], C_TERM)
  ax.set_title(f'row k = {k0}:  $x[{k0}]\\,{sym}[{shift(k0)}]$  (one term, over n)')
  plt.setp(ax.get_xticklabels(), visible=False)

  # column sums = output
  ax = axD_out
  frame(ax, N, ylim_for(y), 'n  (output index)')
  stem(ax, N, y, C_SUM)
  stem(ax, [n0], [y[ci]], C_TERM, ms=8, lw=2.4, z=5)
  ax.set_title(f'column sums  $\\sum_k M[k,n] = {out}[n]$')

  # one column: product over k, drawn sideways so it lines up with the heatmap rows
  ax = axD_col
  col = M[:, ci]
  ax.axvline(0, color='#c3c2b7', lw=0.8)
  nz = col != 0
  ax.hlines(K[nz], 0, col[nz], color=C_TERM, lw=1.6)
  ax.plot(col[nz], K[nz], 'o', color=C_TERM, ms=5, mec='#fcfcfb', mew=0.8)
  ax.plot(col[~nz], K[~nz], 'o', color=C_GHOST, ms=2.5)
  ax.set_xlim(*ylim_for(M))
  ax.grid(axis='y')
  plt.setp(ax.get_yticklabels(), visible=False)
  ax.set_title(f'column n = {n0}\nsum = {out}[{n0}] = {y[ci]:.3g}', fontsize=9)

  axD_txt.axis('off')
  axD_txt.text(0, 0.9, f'{sigD.value}\n* {kerD.value.split("—")[0].strip()}',
               va='top', fontsize=9, color=C_INK2)
  figD.canvas.draw_idle()


sigD = dropdown(SIGNALS, 'signal')
kerD = dropdown(KERNELS, 'kernel')
kD = w.IntSlider(value=0, min=K[0], max=K[-1], description='row k', layout=w.Layout(width='420px'))
nD = w.IntSlider(value=1, min=N[0], max=N[-1], description='column n', layout=w.Layout(width='420px'))
for wd in (sigD, kerD, kD, nD):
  wd.observe(draw_D, names='value')
draw_D()
display(w.VBox([w.HBox([sigD, kerD]), kD, nD, figD.canvas]))

### Where this leads (2.1.2)

From A–D, with δ replaced by $h$:
* **Decomposition** (A): any input is a sum of $x[k]\delta[n-k]$.
* **LTI** (B): the response to $\delta[n-k]$ is $h[n-k]$ (time invariance). The response to the sum is the sum of
  the scaled responses (linearity). So $y[n] = \sum_k x[k]\,h[n-k]$.
* **Computing one output sample** (C): flip $h$, slide it to $n$, multiply by $x$, sum.
* **The whole operation** (D): a banded matrix acting on the vector $x$.

Try `rectangle` with `u[n]−u[n−3]` in B and in C. The two views reach the same $y[n]$ in different orders.

# 2.1.2 — The convolution sum

$$y[n] \;=\; \sum_{k=-\infty}^{+\infty} x[k]\,h[n-k] \;=\; x[n] * h[n]$$

To compute **one** output sample $y[n]$, work along $k$:

1. **flip:** $h[k] \to h[-k]$ (row 1),
2. **slide:** shift the flipped copy so its origin is at $k = n$, giving $h[n-k]$ (row 2, orange, on top of $x[k]$),
3. **multiply** the two sequences sample by sample, $x[k]\,h[n-k]$ (row 3),
4. **sum** over $k$. That number is $y[n]$ (row 4).

Press play and $h$ slides across $x$. The shaded band is where the two sequences **overlap**. Only those $k$
contribute. The output is zero until they start to overlap, grows while the overlap grows, and stops once they
no longer overlap. That gives the case analysis in Oppenheim's worked examples ($n<0$, $0\le n\le 4$, …): each case is a
range of $n$ over which the overlap interval has the same form.

Type your own sequences as `values @ start index`, e.g. `0.5, 2 @ 0` means $x[0]=0.5,\ x[1]=2$.
For the other reading of the same sum (each input sample launches a scaled, shifted copy of $h$), use view **B** above with an $h$ kernel.

In [9]:
def parse_seq(text):
  # '0.5, 2 @ 0' -> (array([0.5, 2.]), 0)
  vals, _, start = text.partition('@')
  vals = np.array([float(v) for v in vals.replace(',', ' ').split()])
  if vals.size == 0:
    raise ValueError('need at least one value')
  return vals, int(start.strip() or 0)


def seq_text(vals, start=0):
  return ', '.join(f'{v:.4g}' for v in vals) + f' @ {start}'


a = 0.8
CONV_PRESETS = {
    'Ex. 2.1 — x = 0.5δ[n]+2δ[n−1], h = 1 on 0..2':
        ('0.5, 2 @ 0', '1, 1, 1 @ 0'),
    'Ex. 2.3 — x = 0.8ⁿu[n], h = u[n]  (both cut at n=9: y exact for n≤9)':
        (seq_text(a ** np.arange(10)), seq_text(np.ones(10))),
    'Ex. 2.4 — x = 1 on 0..4, h = 1.2ⁿ on 0..6':
        (seq_text(np.ones(5)), seq_text(1.2 ** np.arange(7))),
    'rect * rect  →  triangle':
        ('1 1 1 1 @ 0', '1 1 1 1 @ 0'),
    'first difference of a ramp':
        ('0 1 2 3 4 5 @ 0', '1 -1 @ 0'),
    'echo: h = δ[n] + 0.5δ[n−3]':
        ('1 2 1 @ -1', '1 0 0 0.5 @ 0'),
    'non-causal 3-pt average':
        ('0 0 1 1 1 1 1 1 0 0 @ -3', '0.333 0.333 0.333 @ -1'),
}

figE = interactive_fig(figsize=(9, 9))
axE = figE.subplots(4, 1)
figE.subplots_adjust(left=0.08, right=0.97, top=0.95, bottom=0.06, hspace=0.62)

stateE = {}


def setup_E(*_):
  # parse inputs, precompute everything that doesn't depend on n
  try:
    xv, x0 = parse_seq(xE.value)
    hv, h0 = parse_seq(hE.value)
  except ValueError as e:
    msgE.value = f'<span style="color:#d03b3b">can\'t parse input: {e}</span>'
    return
  msgE.value = ''
  xe, he = x0 + xv.size - 1, h0 + hv.size - 1
  y0, ye = x0 + h0, xe + he
  n_lo, n_hi = y0 - 2, ye + 2
  # one shared axis wide enough for x[k], h[k], h[-k] and h[n-k] at both ends of the slide
  A = np.arange(min(x0, h0, -he, n_lo - he), max(xe, he, -h0, n_hi - h0) + 1)

  def seq(vals, start):
    return lambda m: np.where((m >= start) & (m < start + vals.size),
                              vals[np.clip(m - start, 0, vals.size - 1)], 0.0)

  x, h = seq(xv, x0), seq(hv, h0)
  y = np.array([np.sum(x(A) * h(m - A)) for m in A])
  assert np.allclose(y[(A >= y0) & (A <= ye)], np.convolve(xv, hv))
  stateE.update(x=x, h=h, A=A, y=y, x0=x0, xe=xe, h0=h0, he=he)

  nE.unobserve(draw_E, names='value')
  for wd in (nE, playE):   # widen first so min never exceeds max mid-update
    wd.max = max(wd.max, n_hi)
    wd.min, wd.max = n_lo, n_hi
  nE.value = playE.value = n_lo
  nE.observe(draw_E, names='value')
  draw_E()


def draw_E(*_):
  s = stateE
  x, h, A, y = s['x'], s['h'], s['A'], s['y']
  n0 = nE.value
  xk, hk, hflip, hslide = x(A), h(A), h(-A), h(n0 - A)
  prod = xk * hslide
  lo, hi = max(s['x0'], n0 - s['he']), min(s['xe'], n0 - s['h0'])   # overlap interval in k
  lim = ylim_for(xk, hk)

  for ax in axE:
    ax.clear()
    frame(ax, A, lim)

  def shade(ax):
    if lo <= hi:
      ax.axvspan(lo - 0.45, hi + 0.45, color='#1baf7a', alpha=0.09, zorder=0)

  ax = axE[0]
  stem(ax, A + 0.12, hflip, C_H)
  stem(ax, A - 0.12, hk, '#f5b89f', ms=4, lw=1.2, z=2)
  ax.set_title('1. flip:   h[k]  (light)   →   h[−k]  (orange)')

  ax = axE[1]
  shade(ax)
  stem(ax, A - 0.12, xk, C_X)
  stem(ax, A + 0.12, hslide, C_H)
  ax.axvline(n0, color=C_H, lw=0.8, ls=':', zorder=1)
  ax.set_title(f'2. slide:   x[k] (blue)   and   h[{flipped(n0)}] (orange)   — origin of flipped h at k = n = {n0}')

  ax = axE[2]
  ax.set_ylim(*ylim_for(prod, xk * hk.max(), xk * hk.min()))
  shade(ax)
  stem(ax, A, prod, C_TERM)
  if lo <= hi:
    ks = np.arange(lo, hi + 1)
    terms = [f'{float(x(k)):.3g}·{float(h(n0 - k)):.3g}' for k in ks]
    expr = ' + '.join(terms) if len(terms) <= 6 else f'{terms[0]} + … + {terms[-1]}  ({len(terms)} terms)'
    ax.set_title(f'3. multiply:   x[k] h[{flipped(n0)}]   nonzero only on overlap  k ∈ [{lo}, {hi}]')
  else:
    expr = 'no overlap'
    ax.set_title('3. multiply:   no overlap, the product is zero everywhere')
  ax.set_xlabel('k')

  ax = axE[3]
  ax.set_ylim(*ylim_for(y))
  stem(ax, A, y, C_GHOST, ms=3.5, lw=1, z=2)
  stem(ax, A[A <= n0], y[A <= n0], C_SUM)
  stem(ax, [n0], [prod.sum()], C_TERM, ms=8, lw=2.4, z=5)
  ax.set_title(f'4. sum:   y[{n0}] = {expr} = {prod.sum():.4g}')
  ax.set_xlabel('n')
  figE.canvas.draw_idle()


def load_preset(*_):
  xs, hs = CONV_PRESETS[presetE.value]
  for t in (xE, hE):
    t.unobserve(setup_E, names='value')
  xE.value, hE.value = xs, hs
  for t in (xE, hE):
    t.observe(setup_E, names='value')
  setup_E()


presetE = w.Dropdown(options=list(CONV_PRESETS), description='preset', layout=w.Layout(width='620px'))
xE = w.Text(description='x[n]', continuous_update=False, layout=w.Layout(width='620px'))
hE = w.Text(description='h[n]', continuous_update=False, layout=w.Layout(width='620px'))
msgE = w.HTML()
playE, nE = play_slider('n', -2, 2, -2, interval=800)
presetE.observe(load_preset, names='value')
nE.observe(draw_E, names='value')
load_preset()
display(w.VBox([presetE, xE, hE, msgE, w.HBox([playE, nE]), figE.canvas]))

**Things to try**
* `Ex. 2.1`: the output support is $[x_0 + h_0,\; x_{end} + h_{end}]$, so its length is $L_x + L_h - 1$.
* `rect * rect`: the overlap grows linearly and then shrinks, which gives a triangle.
* `Ex. 2.4`: step through $n$ and write down the overlap interval $[\max(0, n-6),\ \min(4, n)]$ for each case, as the book does.
* Swap the `x[n]` and `h[n]` text boxes. The output is identical (**commutativity**), even though the flip-and-slide steps look different.
* `echo` and `first difference`: think of $h$ as a recipe that says which past inputs to add up, and with what weights.